# Query Refactoring

## Overview
The same question can be written many ways in SQL. Refactoring means rewriting a query to be more readable, maintainable, or performant -- without changing the result.

**The three main structural choices:**

| Pattern | When to use | Tradeoffs |
|---|---|---|
| **Correlated subquery** | Simple scalar lookups; small tables | Re-executes per row; slow at scale |
| **CTE (WITH clause)** | Multi-step logic; reused intermediate results; readability | Planner may or may not materialise; excellent for maintainability |
| **Temp table / materialised CTE** | Large intermediate results referenced many times | Explicit materialisation; locks in the intermediate result |

**CTE materialisation in PostgreSQL:**
- PostgreSQL 12 and earlier: CTEs were always materialised (executed once, stored)
- PostgreSQL 12+: the planner may inline CTEs as subqueries if referenced only once
- Force materialisation: `WITH cte AS MATERIALIZED (...)` 
- Force inlining: `WITH cte AS NOT MATERIALIZED (...)`

**General refactoring heuristics:**
1. Push filters as early (deep) as possible -- filter before joining
2. Pre-aggregate before joining when combining two fact tables
3. Replace correlated subqueries with JOINs for large tables
4. Use CTEs to name and isolate logical steps
5. Avoid `SELECT *` in production -- specify only needed columns

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE sites (site_id INTEGER PRIMARY KEY, site_name TEXT NOT NULL, region TEXT, latitude REAL, longitude REAL, elevation_m REAL, established_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE species (species_id INTEGER PRIMARY KEY, common_name TEXT NOT NULL, scientific_name TEXT NOT NULL UNIQUE, taxon_group TEXT, native INTEGER DEFAULT 1, at_risk INTEGER DEFAULT 0);CREATE TABLE field_crew (crew_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, role TEXT, certified INTEGER DEFAULT 1);CREATE TABLE observations (obs_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), species_id INTEGER REFERENCES species(species_id), crew_id INTEGER REFERENCES field_crew(crew_id), obs_date TEXT NOT NULL, count_individuals INTEGER, life_stage TEXT, method TEXT, notes TEXT);CREATE TABLE water_quality (wq_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), crew_id INTEGER REFERENCES field_crew(crew_id), sample_date TEXT NOT NULL, ph REAL, dissolved_o2 REAL, turbidity_ntu REAL, temp_c REAL, conductivity_us REAL);INSERT INTO sites VALUES (1,'Fundy Estuary','Atlantic',45.78,-64.52,3.2,'2018-04-01',1),(2,'Kejimkujik Lake','Atlantic',44.43,-65.21,156.0,'2017-06-15',1),(3,'Presqu ile Point','Great Lakes',43.99,-77.72,75.5,'2019-03-20',1),(4,'Rondeau Bay','Great Lakes',42.31,-81.87,176.0,'2018-09-10',1),(5,'Athabasca Delta','Boreal',58.72,-110.87,213.0,'2016-01-15',1),(6,'Wapusk Tundra','Boreal',57.92,-93.15,45.0,'2017-07-01',1),(7,'Clayoquot Sound','Pacific',49.15,-125.93,12.0,'2015-05-20',1),(8,'Boundary Bay','Pacific',49.01,-122.97,1.5,'2016-08-01',1),(9,'Lac Saint-Pierre','St Lawrence',46.19,-72.87,8.0,'2018-02-14',1),(10,'Baie des Chaleurs','Atlantic',48.01,-65.72,5.0,'2019-11-01',0);INSERT INTO species VALUES (1,'Atlantic Salmon','Salmo salar','Fish',1,1),(2,'Great Blue Heron','Ardea herodias','Bird',1,0),(3,'Wood Duck','Aix sponsa','Bird',1,0),(4,'Spotted Turtle','Clemmys guttata','Reptile',1,1),(5,'Common Loon','Gavia immer','Bird',1,0),(6,'Muskrat','Ondatra zibethicus','Mammal',1,0),(7,'Northern Pike','Esox lucius','Fish',1,0),(8,'Bald Eagle','Haliaeetus leucocephalus','Bird',1,0),(9,'American Bittern','Botaurus lentiginosus','Bird',1,1),(10,'Snapping Turtle','Chelydra serpentina','Reptile',1,0),(11,'Rainbow Trout','Oncorhynchus mykiss','Fish',0,0),(12,'Ring-necked Duck','Aythya collaris','Bird',1,0),(13,'Beaver','Castor canadensis','Mammal',1,0),(14,'River Otter','Lontra canadensis','Mammal',1,0),(15,'Painted Turtle','Chrysemys picta','Reptile',1,0);INSERT INTO field_crew VALUES (1,'Dr. Aroha Ngata','Biologist',1),(2,'Liam Chen','Technician',1),(3,'Fatima Al-Rashid','Biologist',1),(4,'James MacLeod','Technician',1),(5,'Sofia Petrov','Student',0);INSERT INTO observations VALUES (1,1,1,1,'2023-04-10',12,'Adult','Electrofishing',NULL),(2,1,2,2,'2023-04-10',3,'Adult','Visual',NULL),(3,2,5,1,'2023-04-15',2,'Adult','Auditory',NULL),(4,2,4,3,'2023-04-15',8,'Juvenile','Visual',NULL),(5,3,2,2,'2023-05-01',5,'Adult','Visual',NULL),(6,3,3,4,'2023-05-01',7,'Adult','Visual',NULL),(7,4,10,3,'2023-05-10',2,'Adult','Visual',NULL),(8,4,7,1,'2023-05-10',4,'Adult','Electrofishing',NULL),(9,5,13,2,'2023-05-20',6,'Adult','Camera Trap',NULL),(10,5,14,3,'2023-05-20',1,'Adult','Visual',NULL),(11,6,8,1,'2023-06-01',3,'Adult','Visual',NULL),(12,6,6,4,'2023-06-01',9,'Adult','Trap',NULL),(13,7,14,3,'2023-06-15',2,'Adult','Visual',NULL),(14,7,5,2,'2023-06-15',4,'Adult','Auditory',NULL),(15,8,2,1,'2023-07-01',12,'Adult','Visual',NULL),(16,8,8,4,'2023-07-01',2,'Adult','Visual',NULL),(17,9,1,3,'2023-07-10',7,'Adult','Electrofishing',NULL),(18,9,9,1,'2023-07-10',1,'Adult','Visual','First sighting this season'),(19,1,4,2,'2023-08-05',1,'Adult','Visual',NULL),(20,2,13,4,'2023-08-05',4,'Adult','Camera Trap',NULL),(21,3,6,3,'2023-08-12',11,'Adult','Trap',NULL),(22,4,2,2,'2023-08-12',7,'Adult','Visual',NULL),(23,5,8,1,'2023-09-01',1,'Adult','Visual',NULL),(24,6,14,3,'2023-09-01',3,'Adult','Visual',NULL),(25,7,1,4,'2023-09-15',18,'Adult','Electrofishing',NULL),(26,8,5,2,'2023-09-15',6,'Adult','Visual',NULL),(27,9,4,3,'2023-09-20',2,'Juvenile','Visual',NULL),(28,1,7,1,'2023-10-01',3,'Adult','Electrofishing',NULL),(29,2,1,2,'2023-10-01',9,'Adult','Electrofishing',NULL),(30,3,8,4,'2023-10-10',4,'Adult','Visual',NULL),(31,4,5,1,'2023-10-15',3,'Adult','Auditory',NULL),(32,5,2,3,'2023-10-15',8,'Adult','Visual',NULL);INSERT INTO water_quality VALUES (1,1,1,'2023-04-10',7.2,9.1,3.4,8.5,142.0),(2,1,2,'2023-06-15',7.4,8.6,4.2,14.2,138.0),(3,1,3,'2023-08-20',7.1,7.8,5.1,19.6,145.0),(4,2,1,'2023-04-15',6.8,10.2,1.2,7.1,98.0),(5,2,4,'2023-07-01',6.9,9.4,1.8,16.3,102.0),(6,3,2,'2023-05-01',7.6,9.8,2.3,11.4,188.0),(7,3,3,'2023-08-01',7.5,8.2,3.7,20.1,192.0),(8,4,1,'2023-05-10',7.8,9.5,1.9,13.0,205.0),(9,4,4,'2023-09-05',7.7,8.9,2.5,18.4,198.0),(10,5,2,'2023-05-20',7.3,10.8,0.8,9.2,55.0),(11,5,1,'2023-09-10',7.2,9.6,1.1,15.7,58.0),(12,6,3,'2023-06-01',6.5,11.2,0.5,6.8,32.0),(13,7,2,'2023-06-15',7.9,9.0,1.4,12.5,310.0),(14,8,4,'2023-07-01',7.8,8.7,2.1,17.2,295.0),(15,9,1,'2023-07-10',7.6,9.3,2.8,14.8,178.0);""")
conn.commit()
print("Ecological schema ready.")
for t in ["sites","species","field_crew","observations","water_quality"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Ecological schema ready.
  sites: 10 rows
  species: 15 rows
  field_crew: 5 rows
  observations: 32 rows
  water_quality: 15 rows


---
## Correlated subquery vs CTE: site summary

In [2]:
import pandas as pd
q_nested = """
SELECT  s.site_name,
        s.region,
        (SELECT COUNT(*) FROM observations o
         WHERE o.site_id = s.site_id)  AS observations,
        (SELECT COUNT(DISTINCT o2.species_id) FROM observations o2
         WHERE o2.site_id = s.site_id) AS species_richness,
        (SELECT ROUND(AVG(wq.ph),2) FROM water_quality wq
         WHERE wq.site_id = s.site_id) AS avg_ph
FROM    sites AS s
WHERE   s.active = 1
ORDER BY observations DESC
"""
print("=== Approach 1: correlated scalar subqueries (one sub-query per row per column) ===")
df1 = pd.read_sql(q_nested, conn)
print(df1.to_string(index=False))
print()
print("Problem: with N active sites and 3 correlated subqueries,")
print("the database executes up to 3*N separate sub-queries.")
print("On 1 million sites this is 3 million extra queries.")

=== Approach 1: correlated scalar subqueries (one sub-query per row per column) ===
       site_name      region  observations  species_richness  avg_ph
   Fundy Estuary    Atlantic             4                 4    7.23
 Kejimkujik Lake    Atlantic             4                 4    6.85
Presqu ile Point Great Lakes             4                 4    7.55
     Rondeau Bay Great Lakes             4                 4    7.75
 Athabasca Delta      Boreal             4                 4    7.25
   Wapusk Tundra      Boreal             3                 3    6.50
 Clayoquot Sound     Pacific             3                 3    7.90
    Boundary Bay     Pacific             3                 3    7.80
Lac Saint-Pierre St Lawrence             3                 3    7.60

Problem: with N active sites and 3 correlated subqueries,
the database executes up to 3*N separate sub-queries.
On 1 million sites this is 3 million extra queries.


---
## Refactored: CTE pre-aggregation

In [3]:
import pandas as pd
q_cte = """
WITH site_obs AS (
    SELECT  site_id,
            COUNT(*)               AS obs_count,
            SUM(count_individuals) AS total_individuals,
            COUNT(DISTINCT species_id) AS species_richness
    FROM    observations
    GROUP BY site_id
),
site_wq AS (
    SELECT  site_id,
            ROUND(AVG(ph), 2)           AS avg_ph,
            ROUND(AVG(dissolved_o2), 2) AS avg_do,
            ROUND(AVG(temp_c), 2)       AS avg_temp_c,
            COUNT(*)                    AS wq_samples
    FROM    water_quality
    GROUP BY site_id
),
at_risk_obs AS (
    SELECT  o.site_id,
            COUNT(DISTINCT o.species_id) AS at_risk_species
    FROM    observations AS o
    JOIN    species      AS sp ON o.species_id = sp.species_id
    WHERE   sp.at_risk = 1
    GROUP BY o.site_id
)
SELECT  s.site_name,
        s.region,
        COALESCE(so.obs_count,         0) AS observations,
        COALESCE(so.species_richness,  0) AS species_richness,
        COALESCE(so.total_individuals, 0) AS total_individuals,
        COALESCE(ar.at_risk_species,   0) AS at_risk_species,
        wq.avg_ph,
        wq.avg_do,
        wq.avg_temp_c,
        COALESCE(wq.wq_samples,        0) AS wq_samples
FROM    sites       AS s
LEFT JOIN site_obs  AS so ON s.site_id = so.site_id
LEFT JOIN site_wq   AS wq ON s.site_id = wq.site_id
LEFT JOIN at_risk_obs AS ar ON s.site_id = ar.site_id
WHERE   s.active = 1
ORDER BY so.obs_count DESC NULLS LAST
"""
print("=== Approach 2: CTEs pre-aggregate each table once, then join ===")
df2 = pd.read_sql(q_cte, conn)
print(df2.to_string(index=False))
print()
print("Each CTE runs once. The final query is three LEFT JOINs.")
print("Scales to millions of rows; no per-row sub-execution.")

=== Approach 2: CTEs pre-aggregate each table once, then join ===
       site_name      region  observations  species_richness  total_individuals  at_risk_species  avg_ph  avg_do  avg_temp_c  wq_samples
   Fundy Estuary    Atlantic             4                 4                 19                2    7.23     8.5       14.10           3
 Kejimkujik Lake    Atlantic             4                 4                 23                2    6.85     9.8       11.70           2
Presqu ile Point Great Lakes             4                 4                 27                0    7.55     9.0       15.75           2
     Rondeau Bay Great Lakes             4                 4                 16                0    7.75     9.2       15.70           2
 Athabasca Delta      Boreal             4                 4                 16                0    7.25    10.2       12.45           2
   Wapusk Tundra      Boreal             3                 3                 15                0    6.50    11.2

---
## When to use temp tables vs CTEs

In [4]:
print("=== Temp table: materialise an expensive intermediate result ===")
# SQLite temp tables
conn.executescript("""
    CREATE TEMP TABLE tmp_site_obs AS
    SELECT  site_id,
            COUNT(*)               AS obs_count,
            SUM(count_individuals) AS total_individuals,
            COUNT(DISTINCT species_id) AS species_richness
    FROM    observations
    GROUP BY site_id;

    CREATE INDEX tmp_idx_site ON tmp_site_obs (site_id);
""")
conn.commit()

import pandas as pd
print("Temp table contents (materialised once, indexed):")
print(pd.read_sql("SELECT * FROM tmp_site_obs ORDER BY obs_count DESC", conn).to_string(index=False))

print()
print("Temp table vs CTE decision guide:")
rows = [
    ("CTE used once",                     "CTE",        "Planner inlines it; no materialisation cost"),
    ("CTE referenced 2+ times",           "Temp table", "CTE may re-execute; temp table runs once"),
    ("Intermediate result is small",      "CTE",        "Overhead of temp table not worth it"),
    ("Intermediate result is large",      "Temp table", "Add index; reuse multiple times cheaply"),
    ("In a stored proc / multi-step ETL", "Temp table", "Survives across multiple queries in session"),
    ("One-off analytical query",          "CTE",        "More readable; no cleanup needed"),
]
print(f"  {"Scenario":<40s}  {"Prefer":<12s}  Reason")
print("  " + "-"*80)
for scenario, prefer, reason in rows:
    print(f"  {scenario:<40s}  {prefer:<12s}  {reason}")

=== Temp table: materialise an expensive intermediate result ===
Temp table contents (materialised once, indexed):
 site_id  obs_count  total_individuals  species_richness
       1          4                 19                 4
       2          4                 23                 4
       3          4                 27                 4
       4          4                 16                 4
       5          4                 16                 4
       6          3                 15                 3
       7          3                 24                 3
       8          3                 20                 3
       9          3                 10                 3

Temp table vs CTE decision guide:
  Scenario                                  Prefer        Reason
  --------------------------------------------------------------------------------
  CTE used once                             CTE           Planner inlines it; no materialisation cost
  CTE referenced 2+ times     

---
## Readability refactoring: flatten deep nesting

In [5]:
import pandas as pd
print("=== Before: deeply nested subquery ===")
ugly = """
SELECT region, avg_individuals
FROM (
    SELECT s.region,
           ROUND(AVG(sub.site_total), 1) AS avg_individuals
    FROM sites AS s
    JOIN (
        SELECT site_id, SUM(count_individuals) AS site_total
        FROM observations
        GROUP BY site_id
    ) AS sub ON s.site_id = sub.site_id
    GROUP BY s.region
) AS outer_q
ORDER BY avg_individuals DESC
"""
print(ugly)
df_ugly = pd.read_sql(ugly, conn)

print("=== After: same logic as flat CTEs ===")
clean = """
WITH site_totals AS (
    SELECT site_id, SUM(count_individuals) AS site_total
    FROM   observations
    GROUP BY site_id
),
region_averages AS (
    SELECT s.region,
           ROUND(AVG(st.site_total), 1) AS avg_individuals
    FROM   sites AS s
    JOIN   site_totals AS st ON s.site_id = st.site_id
    GROUP BY s.region
)
SELECT region, avg_individuals
FROM   region_averages
ORDER BY avg_individuals DESC
"""
print(clean)
df_clean = pd.read_sql(clean, conn)

import pandas as pd
print("Results match:", df_ugly.equals(df_clean))
print(df_clean.to_string(index=False))
conn.close()

=== Before: deeply nested subquery ===

SELECT region, avg_individuals
FROM (
    SELECT s.region,
           ROUND(AVG(sub.site_total), 1) AS avg_individuals
    FROM sites AS s
    JOIN (
        SELECT site_id, SUM(count_individuals) AS site_total
        FROM observations
        GROUP BY site_id
    ) AS sub ON s.site_id = sub.site_id
    GROUP BY s.region
) AS outer_q
ORDER BY avg_individuals DESC

=== After: same logic as flat CTEs ===

WITH site_totals AS (
    SELECT site_id, SUM(count_individuals) AS site_total
    FROM   observations
    GROUP BY site_id
),
region_averages AS (
    SELECT s.region,
           ROUND(AVG(st.site_total), 1) AS avg_individuals
    FROM   sites AS s
    JOIN   site_totals AS st ON s.site_id = st.site_id
    GROUP BY s.region
)
SELECT region, avg_individuals
FROM   region_averages
ORDER BY avg_individuals DESC

Results match: True
     region  avg_individuals
    Pacific             22.0
Great Lakes             21.5
   Atlantic             21.0
  

---
## Common Pitfalls

**1. Assuming CTEs are always materialised (or always inlined)**
In PostgreSQL 11 and earlier, every CTE was materialised as a temporary result. In PostgreSQL 12+, the planner may inline a CTE used only once. Do not rely on CTE materialisation as a performance strategy unless you use the explicit `WITH cte AS MATERIALIZED (...)` syntax. Test with EXPLAIN ANALYZE.

**2. Referencing a CTE multiple times without realising it re-executes**
In PostgreSQL 12+ with inlining enabled, a CTE referenced three times in the final query may run three times -- the same as writing the subquery inline three times. If the CTE is expensive, force materialisation or move it to a temp table.

**3. Correlated subqueries in SELECT for large tables**
`SELECT (SELECT COUNT(*) FROM observations WHERE site_id = s.site_id) FROM sites` executes one COUNT query per site row. For 10 sites this is fine; for 10 million sites it executes 10 million correlated queries. Always replace correlated subqueries with a pre-aggregating JOIN or CTE when the outer table is large.

**4. Refactoring changes the result due to implicit DISTINCT or filter placement**
Moving a WHERE filter from inside a subquery to outside a CTE can change results if NULLs or aggregation is involved. Always verify result equality before and after refactoring, especially when removing DISTINCT, changing join types from INNER to LEFT, or relocating HAVING vs WHERE.

**5. Deep nesting as a readability debt accumulator**
A query with 5 levels of nested subqueries is technically correct but essentially unreadable and unmaintainable. Refactoring to named CTEs has zero performance cost in most cases and makes the query self-documenting. Treat deep nesting the same way you treat deeply nested code: a signal to refactor.

**6. Using SELECT * in CTEs and subqueries**
`WITH base AS (SELECT * FROM observations)` passes every column downstream, including columns that downstream CTEs never use. This wastes memory in materialised CTEs and makes it impossible to tell at a glance what data each step consumes. Always project only the columns needed at each step.


---
*sql_methods_library - Samantha McGarrigle*